In [26]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os
import urllib.request
import zipfile
import numpy as np


print("TensorFlow versione:", tf.__version__)

TensorFlow versione: 2.20.0


In [27]:
# Scaricare i dataset
print("Scaricando training set...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/rps.zip",
    "rps.zip"
)

print("Scaricando test set...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip",
    "rps-test-set.zip"
)

print("Estraendo...")
with zipfile.ZipFile("rps.zip", "r") as z:
    z.extractall(".")
with zipfile.ZipFile("rps-test-set.zip", "r") as z:
    z.extractall(".")

print("Dataset pronti!")

Scaricando training set...
Scaricando test set...
Estraendo...
Dataset pronti!


In [28]:
# Percorsi delle cartelle
TRAIN_DIR = "./rps"
TEST_DIR  = "./rps-test-set"

# Generatore TRAINING con augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalizzazione
    rotation_range=40,       # Ruota fino a 40 gradi
    width_shift_range=0.2,   # Sposta orizzontalmente
    shear_range=0.2,         # Deforma
    horizontal_flip=True,    # Specchia
    fill_mode='nearest'      # Riempie i pixel vuoti
)

# Generatore TEST (solo normalizzazione, niente augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

# Collega i generatori alle cartelle
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(150, 150),  # Ridimensiona ogni immagine a 150x150
    batch_size=32,
    class_mode='categorical' # 3 classi: rock, paper, scissors
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

print("Classi trovate:", train_generator.class_indices)

Found 2520 images belonging to 3 classes.
Found 372 images belonging to 3 classes.
Classi trovate: {'paper': 0, 'rock': 1, 'scissors': 2}


In [ ]:
# Alcune immagini per visualizzare il processo
sample_images, sample_labels = next(train_generator)

plt.figure(figsize=(10, 5))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(sample_images[i])
    plt.axis('off')
plt.suptitle("Esempi di immagini con Augmentation")
plt.show()

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([

    # --- BLOCCO 1 ---
    # 32 filtri cercano pattern semplici (bordi, linee)
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    # --- BLOCCO 2 ---
    # 64 filtri cercano pattern più complessi (curve, angoli)
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- BLOCCO 3 ---
    # 128 filtri cercano pattern ancora più complessi (forme di dita)
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- BLOCCO 4 ---
    # 128 filtri per affinare ulteriormente
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- PARTE FINALE ---
    Flatten(),           # Srotola tutto in una lista
    Dense(512, activation='relu'),  # 512 neuroni per "ragionare"
    Dense(3, activation='softmax')  # 3 neuroni = 3 classi finali
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,147 (13.18 MB)

 Trainable params: 3,454,147 (13.18 MB)

 Non-trainable params: 0 (0.00 B)